# 8주차 실습 — SQL (2) 집계와 JOIN

**오늘 끝나면 할 수 있어야 하는 것**

- ① 집계 함수로 통계를 구할 수 있다
- ② GROUP BY로 묶어 세고, HAVING으로 그룹을 거를 수 있다
- ③ 두 표를 JOIN으로 연결하고, 행 수 변화를 확인할 수 있다

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` 같은 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNqdlt9v2lYUx/+VK55AiiLAmB+T9tBlrM20kTVNN1XTFNlgFqQ0iYBsqqpKpHM2EiIV2hDIBJSupKEV0rzgNEyim7Q/x/fwP+yca0wMSx6yJ9vnnnPs87nfc64fe+4GPB+xx55c5nu8emRZldWIlpKUQMozxzyb19izmz/m0B7A2+TmOt1+64G2DmcmrfJS1+rr0CpC88D1/NsuP6l7vkPDWmYjT2kXlu4nVrxQM33QLDDLeA5Nk2Ec3z9ko6M9fnowx2yfTxfvrSwmFlaY4wx6g79qMF7q8JOBHVrHSwOKdX4yZLC7x0tFXJ0XZWRTWlZL4SvTynpOI8tWPrO5oaw7pidouxt0o0iG5UgoFkgGNDniRjFrvwrF6NmeddGmVehUoHqMnzdVdwDaRTaqNKBWxjqY7c7+uWATd2jqzDrvjap1waLadeoy/oKqzvsm6AP49XDeiYVWmb/pwXHhMsXYZhkFxo1DXjiAU1yt7oPIiQ/8mc64uXtzTpKbU1STJTkSDKmyprg5zdodTv4pzZQa0KpAzcBvISfrDxMLnWLFn7bhtEG1WxcDflAQu957D40hf2simQYphXRDrF53Rbnlt+Oi2IP4rWWvncJHkfarUFiMf0BO+jjVmOYNGITcDNSkpIT9MjZHRHYzCEXVgJSWI/60bXcYRN0Mcvnt1KPV/KMtTYT+oGWV9fXVXF7Jb+euYXIv/kV8YWWsHjitYHc5GsEaLaMMpYGz29QrNZMu/F2XV8rs9vLS/a/YJw8wnBQA7R3YP7d5MS9qBl4P2Vh3UOpgThbw3YSM7CaTlpBKUJJVNeZ3k1GiajoZS0dSqpR0kZHcYLbWlJx2nSpw245MUb6OyjgzrP7QK0aFD+tiqBNSBrwyKBZr/+ZOfDlO2qDuwVqbQ3bn1teLidvkDYc7jl5E1N9D6iVcEFHUVGNfbEpeqdPQYfBiwIgNTTkH3zyzBgZ0CvTC4Jzf78eXM5Qm/LQDrf/RZ2E3SUkOK4FkSvGHNMlNctZ+5WhuDnl/gM02qtX/QxIlgaOItnsjmV/NpJjok9YvrrLsBXIZewvGtpwIwJHJSMUZLTdvu8JZnb8ckifVz9/oAt3740sMNmn7i2jo13r8ZY/4WWZhImydDoFynf8uut5eJOrTG2lnYfysOKoPxJDsVHj/HHP+LFJjb9+4vSNu9KlwNB32x6RkIDx1FEihdEwLqulYLBhyoZfd6B9qubXVvJZ9KE6DK3cATouj4zrv6wK8PuDtDo022zzmRz3sneTyOUPwcos+X1pMkO4+W176Usx78njemZu0O54ZQrfVlpC8q+tnuOSz29diiU6dkJKkqmpIkqLRqZ+FWfuVirySBBXh/ZgtJhLxZVGRfdQfVeDdh2nJXf4hMP60a/05aWA4HqKzdyxHn2vukd/AoECbFTJ9YdxMGU/+BcGmGZI=")   # 자동 확인용 (정답 SQL 은 없음)

## 1. 집계로 통계 내기 — COUNT · AVG · 중앙값 · 비율

### 집계 함수 — 다섯 개면 대부분의 통계가 됩니다

| 함수 | 무엇을 구하나 | 예 | NULL은 |
|---|---|---|---|
| `COUNT` | 개수 | `COUNT(*)` | `COUNT(*)`는 포함, `COUNT(열)`은 제외 |
| `SUM` | 합계 | `SUM(enrollment)` | 제외하고 더함 |
| `AVG` | 평균 | `AVG(enrollment)` | 제외하고 나눔 |
| `MIN` / `MAX` | 가장 작은 / 큰 값 | `MAX(start_date)` | 제외 |
| `MEDIAN` | 중앙값 | `MEDIAN(enrollment)` | 제외 |

**주의**: 집계 함수는 NULL을 빼고 계산함.

### COUNT — COUNT(*) · COUNT(열) · COUNT(DISTINCT 열)

COUNT는 세 가지 모양이 있고, 결과가 서로 다릅니다.

| 쓰는 법 | 무엇을 세나 | studies에서 | 예시 5행 결과 | 주의 |
|---|---|---:|---:|---|
| `COUNT(*)` | 모든 행 | 37,509 | 5 | NULL도 포함해서 셈 |
| `COUNT(열)` | 그 열에 값이 있는 행 | 4,352 | 3 | NULL은 세지 않음 |
| `COUNT(DISTINCT 열)` | 서로 다른 값의 개수 | 9 | 2 | 중복을 뺀 종류 수 |

예시 5행

| | results_first_posted_date | overall_status |
|---|---|---|
| 시험 A | 2023-05-01 | COMPLETED |
| 시험 B | NULL | COMPLETED |
| 시험 C | 2022-11-10 | RECRUITING |
| 시험 D | NULL | RECRUITING |
| 시험 E | 2024-02-03 | COMPLETED |

In [ ]:
%%sql
-- [2번] COUNT 세 가지 모양 → 37,509 / 4,352 / 9
SELECT COUNT(*) AS 전체행,
       COUNT(results_first_posted_date) AS 결과있음,
       COUNT(DISTINCT overall_status) AS 상태종류
FROM studies;

### 평균과 중앙값 — 극단값 하나가 평균을 크게 바꿉니다

In [ ]:
%%sql
-- [3번] “중재 연구의 등록 인원 평균 · 중앙값 · 최대는?”
-- ‘1만 명 이하’ 행: AND enrollment <= 10000
SELECT AVG(enrollment) AS 평균,
       MEDIAN(enrollment) AS 중앙값,
       MAX(enrollment) AS 최대
FROM studies
WHERE study_type = 'INTERVENTIONAL';

| | 평균 (반올림) | 중앙값 | 최대 |
|---|---:|---:|---:|
| 전체 | 1,279 | 60 | 12,317,546 |

### 비율 구하기 — 퍼센트는 100.0 을 곱해서 계산합니다

In [ ]:
%%sql
-- [4번] “완료된 시험 중 결과가 게시된 비율은?” → 18.0
SELECT ROUND(100.0
         * COUNT(results_first_posted_date)
         / COUNT(*), 1) AS 결과게시율
FROM studies
WHERE overall_status = 'COMPLETED';

100.0 을 곱하는 이유: INT와 FLOAT의 차이, PostgreSQL · SQL Server 등은 정수끼리 나누면 소수점이 잘려 0이 됨 (실습 DuckDB 는 소수가 나옴)

**ROUND(숫자, 자릿수)** — 소수점 몇째 자리까지 반올림할지 지정

- 예) `ROUND(3.141, 1)` → 3.1
- 예) `ROUND(3.141, 2)` → 3.14

### 빈 값도 드러내기 — COALESCE

NULL 은 이름을 붙여야 보입니다.

In [ ]:
%%sql
-- [5번] “임상 단계를 보여 줘. 안 적힌 건 ‘미기재’로.”
SELECT phase,
       COALESCE(phase, '미기재') AS 단계
FROM studies;

`COALESCE(값, '대체값')`: 값이 비어 있으면(NULL) 대체값을, 아니면 원래 값을 그대로 보여 줍니다.

## 2. GROUP BY와 HAVING — 묶어서 세고, 묶은 결과를 거르기

### SELECT 문의 절 — 7강의 다섯 절에 GROUP BY · HAVING 두 개가 더해집니다

| 절 | 무엇을 | 예 |
|---|---|---|
| `SELECT` | 무엇을 볼지 (열 · 집계) | `phase, COUNT(*)` |
| `FROM` | 어디에서 (테이블) | `studies` |
| `WHERE` | 어떤 행만 (묶기 전 조건) | `enrollment >= 100` |
| `GROUP BY` **NEW** | 무엇끼리 묶을지 (그룹) | `phase` |
| `HAVING` **NEW** | 어떤 그룹만 (묶은 뒤 조건) | `COUNT(*) >= 10` |
| `ORDER BY` | 어떤 순서로 (정렬) | `COUNT(*) DESC` |
| `LIMIT` | 몇 줄만 (개수) | `10` |

### GROUP BY가 없다면

묶는 기준이 없으면 결과도 한 행

In [ ]:
%%sql
-- [6번] “전체 연구는 몇 건이야?”  (GROUP BY 없음)
SELECT COUNT(*) AS 건수,
       MEDIAN(enrollment) AS 중앙값
FROM studies;

### GROUP BY — 묶어서 세기

**결과 한 행 = 그룹 하나** 입니다.

In [ ]:
%%sql
-- [7번] “연구 유형별 건수와 중앙값은?”
SELECT study_type, COUNT(*) AS 건수,
       MEDIAN(enrollment) AS 중앙값
FROM studies
GROUP BY study_type
ORDER BY 건수 DESC;

묶는 열(`study_type`)은 SELECT에도 적어야 결과를 읽을 수 있습니다.

반대로, SELECT에서 집계 함수로 감싸지 않은 열은 모두 GROUP BY에도 적습니다.

### 두 열로 묶기 — 교차표 만들기

GROUP BY 에 열을 더 적으면 됩니다.

In [ ]:
%%sql
-- [8번] “2023년 이후, 연도별 · 연구 유형별 건수와 중앙값은?”
SELECT YEAR(start_date) AS 연도,
       study_type, COUNT(*) AS 건수,
       MEDIAN(enrollment) AS 중앙값
FROM studies
WHERE start_date >= '2023-01-01'
GROUP BY 연도, study_type
ORDER BY 연도, 건수 DESC;

앞 쿼리(GROUP BY — 묶어서 세기)에 연도 · WHERE · 정렬을 더했습니다.

`YEAR(날짜)` = 날짜에서 연도만 꺼내는 함수.

### HAVING — 집계 결과에 조건

WHERE는 묶기 전, HAVING은 묶은 뒤

In [ ]:
%%sql
-- [9번] “시험이 1만 건 이상인 연구 유형만 보여 줘”
SELECT study_type, COUNT(*) AS 건수,
       MEDIAN(enrollment) AS 중앙값
FROM studies
GROUP BY study_type
HAVING COUNT(*) >= 10000
ORDER BY 건수 DESC;

‘GROUP BY — 묶어서 세기’의 쿼리에 HAVING 한 줄만 더했습니다. 묶은 뒤 그룹마다 센 건수(COUNT)로 그룹을 거릅니다.

연구 유형별로 묶은 결과 → HAVING이 그룹을 거름 (OBSERVATIONAL 9,593건은 제외)

### 적는 순서와 처리 순서

7강의 처리 순서에서 WHERE와 SELECT 사이에 GROUP BY · HAVING이 들어갑니다.

집계에 대한 조건은 WHERE가 아니라 **HAVING**에 씁니다.

| | 순서 |
|---|---|
| 적는 순서 (사람의 순서) | SELECT › FROM › WHERE › GROUP BY › HAVING › ORDER BY › LIMIT |
| 처리 순서 (컴퓨터의 순서) | FROM › WHERE › GROUP BY › HAVING › SELECT › ORDER BY › LIMIT |

**왜 WHERE에는 집계 함수를 못 쓰나** — WHERE는 GROUP BY보다 먼저 처리됨. 아직 묶기 전이라 그룹마다의 COUNT(*)가 없음.

`WHERE COUNT(*) >= 10` → 오류 / `HAVING COUNT(*) >= 10` → 가능

**WHERE와 HAVING은 둘 다 거른다**

- WHERE — 묶기 전에 행을 거름
- HAVING — 묶은 뒤 그룹을 거름
- 한 쿼리에 둘 다 쓸 수 있음. 행을 먼저 거르고, 묶은 결과를 다시 거름.

### GROUP BY · HAVING이 처리되는 과정

행을 묶은 뒤에야 그룹마다 COUNT(*)가 생기고, HAVING은 그 값으로 거릅니다

예시 쿼리 `SELECT study_type, COUNT(*) FROM studies GROUP BY study_type HAVING COUNT(*) >= 3;`

| 단계 | 하는 일 | 예시 6행 (시험 A~F) |
|---|---|---|
| ① `FROM studies` | 표 전체를 가져옴 | 6행 |
| ② `GROUP BY study_type` | 같은 study_type끼리 묶음 | INTERVENTIONAL: A, C, D, F / OBSERVATIONAL: B, E |
| ③ `HAVING COUNT(*) >= 3` | 3건 이상인 그룹만 남김 | INTERVENTIONAL 4 남김 / 2건인 OBSERVATIONAL은 제외 |
| ④ `SELECT study_type, COUNT(*)` | 그룹마다 한 줄로 보여 줌 | INTERVENTIONAL · 4 |

행 6개 → 그룹 2개 → 1개만 남음

WHERE는 ①과 ② 사이에서 행을 거르고, HAVING은 ② 다음에 그룹을 거릅니다.

예: `WHERE enrollment >= 100` 을 더하면 ①의 행 중 100명 이상인 시험만 묶입니다.

### CASE WHEN — 조건에 따라 나누어 세기 (참고만)

엑셀의 IF 와 같은 일을 합니다.

In [ ]:
%%sql
-- [10번] “등록 인원을 소 · 중 · 대규모로 나누어 세어 줘”
SELECT CASE WHEN enrollment < 50 THEN '소규모'
            WHEN enrollment < 500 THEN '중규모'
            ELSE '대규모'
       END AS 규모,
       COUNT(*) AS 건수
FROM studies
WHERE enrollment IS NOT NULL
GROUP BY 규모;

- **WHERE**: 테이블 전체 데이터 중 원하는 조건에 부합하는 행(Row)만 남기고 나머지는 버림.
- **CASE WHEN**: if-else 문처럼 동작하며, 조건에 따라 새로운 값이나 열(Column)을 생성

중간 과정 (예시 5행) — CASE WHEN이 행마다 붙이는 값

| enrollment | 30 | 120 | 800 | 45 | 300 |
|---|---|---|---|---|---|
| → 규모 | 소규모 | 중규모 | 대규모 | 소규모 | 중규모 |

## 3. JOIN — 두 표를 연결하기

### JOIN이 필요한 이유 — 한 표에 없는 정보를 다른 표에서 가져오기

| studies | browse_conditions |
|---|---|
| `nct_id` (PK) | `id` (PK) |
| `brief_title` | `nct_id` (FK) |
| `phase` | `mesh_term` |

`nct_id` 로 연결

**공통으로 가진 열** — 두 표 모두 `nct_id`가 있음

두 표를 잇는 조건은 **ON** 절에 적습니다.

In [ ]:
%%sql
-- [11번] studies 와 browse_conditions 를 nct_id 로 연결
SELECT studies.nct_id, mesh_term
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
LIMIT 4;

질환이 여러 개인 시험은 질환 수만큼 행이 생깁니다.

### JOIN 뒤에는 행 수를 확인

한 시험에 질환이 여러 개 붙어 있습니다.

In [ ]:
%%sql
-- [12번] “JOIN하면 행이 몇 개가 되지?” → 309,124 / 31,634
SELECT COUNT(*) AS JOIN후_행수,
  COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id;

행이 약 10배로 늘었습니다. 이 상태에서 `COUNT(*)`를 세면 시험 수가 아니라 (시험 × 질환) 쌍의 수입니다.

앞 쿼리(JOIN이 필요한 이유)에서 SELECT만 바꿨습니다.

### JOIN에 WHERE 더하기 — 뇌졸중 시험 수 구하기

In [ ]:
%%sql
-- [13번] “뇌졸중 시험은 몇 건이야?” → 1,674
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Stroke';

질환 이름은 `conditions`(연구자 입력)가 아니라 `browse_conditions` 를 씁니다.

MeSH = 의학 주제어 표준 목록. 같은 질환을 한 이름으로 부르기 위한 것입니다 (14주차 표준 용어).

**읽는 법**

- ① `FROM studies`: 시험 표에서 시작
- ② `JOIN browse_conditions`: 질환 표를 붙임
- ③ `ON`: nct_id가 같은 행끼리 연결

앞 쿼리에 WHERE 한 줄을 더했습니다. SELECT는 시험 수만 남겼습니다.

### JOIN 뒤에 GROUP BY — 질환별로 세기

JOIN은 FROM 바로 뒤, GROUP BY보다 앞에 적습니다

In [ ]:
%%sql
-- [14번] “질환별 시험 수는?”
SELECT mesh_term,
       COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
GROUP BY mesh_term
ORDER BY 시험수 DESC;

JOIN으로 붙인 표를 먼저 만들고, 그 표를 GROUP BY로 묶는다고 생각하면 됩니다.

**SQL을 적는 순서**

1. `SELECT` — 무엇을 보여 줄지
2. `FROM` — 시작하는 표
3. `JOIN … ON` — 붙일 표와 연결 조건
4. `WHERE` — 행 거르기
5. `GROUP BY` — 묶기
6. `ORDER BY` — 정렬

앞 쿼리에서 WHERE를 빼고 `mesh_term` · GROUP BY · ORDER BY를 더했습니다.

### JOIN의 두 가지 — 양쪽에 다 있는 것만 vs 왼쪽은 모두

예시: studies 에 시험 A, 시험 B / facilities 에 A–서울병원, A–부산병원 → 시험 B는 기관 정보 없음

| JOIN (= INNER JOIN) | LEFT JOIN |
|---|---|
| 양쪽에 모두 있는 행만 남김 | 왼쪽(studies)은 모두 남김. 오른쪽에 없으면 NULL로 채움 |
| 시험 B가 빠짐 → 실제: 34,810건 | 시험 B는 NULL로 남음 → 실제: 37,509건 |

In [ ]:
%%sql
-- [15번] JOIN → 34,810건
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies JOIN facilities
  ON studies.nct_id = facilities.nct_id;

In [ ]:
%%sql
-- [16번] LEFT JOIN → 37,509건
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies LEFT JOIN facilities
  ON studies.nct_id = facilities.nct_id;

“전체 시험 중 몇 %”를 구할 때 INNER JOIN을 쓰면 **분모가 조용히 줄어듭니다.**

## 4. 자주 하는 실수 — 오류가 나는 실수 · 나지 않는 실수

### 자주 하는 실수 1: GROUP BY에 없는 열

**하고 싶은 일** — 연구 유형별로, 진행 상태마다 시험이 몇 건인지 세기

In [ ]:
%%sql
-- [17번] [오류 예시] 이렇게 쓰면 → Binder Error
SELECT study_type, overall_status,
       COUNT(*)
FROM studies
GROUP BY study_type

In [ ]:
%%sql
-- [18번] 이렇게 고칩니다
SELECT study_type, overall_status,
       COUNT(*)
FROM studies
GROUP BY study_type, overall_status

**왜 오류일까** — `GROUP BY study_type` 로 묶으면 한 묶음에 상태가 여러 개라 무엇을 보여 줄지 정할 수 없습니다.

SELECT에서 집계 함수로 감싸지 않은 열은 모두 GROUP BY에도 적습니다.

### 자주 하는 실수 2: WHERE에 집계 함수

실행하면 오류 메시지가 나옵니다.

In [ ]:
%%sql
-- [19번] [오류 예시] 이렇게 쓰면 → Binder Error
SELECT phase, COUNT(*)
FROM studies
WHERE COUNT(*) > 1000
GROUP BY phase

In [ ]:
%%sql
-- [20번] 이렇게 고칩니다
SELECT phase, COUNT(*)
FROM studies
GROUP BY phase
HAVING COUNT(*) > 1000

WHERE는 묶기 전이라 COUNT를 모릅니다.

집계에 대한 조건은 HAVING에 씁니다.

### 자주 하는 실수 3: JOIN 뒤 COUNT(*)를 시험 수로 착각

오류가 나지 않아 더 위험합니다.

In [ ]:
%%sql
-- [21번] 이렇게 쓰면 → 결과 4,098 — 오류는 없지만 이것은 기관 수입니다
SELECT COUNT(*) AS 한국시험수
FROM studies
JOIN facilities
  ON studies.nct_id = facilities.nct_id
WHERE country = 'South Korea'

In [ ]:
%%sql
-- [22번] 이렇게 고칩니다 → 1,182 = 실제 시험 수
SELECT COUNT(DISTINCT studies.nct_id)
         AS 한국시험수
FROM studies
JOIN facilities
  ON studies.nct_id = facilities.nct_id
WHERE country = 'South Korea'

한 시험이 여러 기관에서 진행되기 때문입니다.

`COUNT(DISTINCT 키)`를 씁니다.

예시: 시험 A는 한국 기관 3곳, 시험 B는 1곳에서 진행 → JOIN하면 행이 4개

- `COUNT(*)` = 4 (기관 수)
- `COUNT(DISTINCT …)` = 2 (시험 수)

## 직접 해보기 — 8문항

`___` 를 채워 실행하면 결과 아래에 ✅ / ❌ 가 나옵니다.

**Q1. COUNT 세 가지** — `phase` 에 **값이 있는** 시험 수(`단계있음`)와 `phase` 값의 **종류 수**(`단계종류`)를 세세요.

In [ ]:
%%sql
-- [23번] Q1
SELECT COUNT(*) AS 전체,
       COUNT(___) AS 단계있음,
       COUNT(___) AS 단계종류
FROM studies;

**Q2. 평균과 중앙값** — 3상(`PHASE3`) 시험의 등록 인원 평균과 중앙값을 구하세요. 두 값이 얼마나 다른지 보세요.

In [ ]:
%%sql
-- [24번] Q2
SELECT ROUND(___(enrollment)) AS 평균,
       ___(enrollment) AS 중앙값
FROM studies
WHERE phase = 'PHASE3';

**Q3. GROUP BY** — 완료(`COMPLETED`)된 시험을 **시작 연도별로** 세세요. 연도는 `YEAR(start_date)` 로 꺼냅니다.

In [ ]:
%%sql
-- [25번] Q3
SELECT ___ AS 시작연도, COUNT(*) AS 건수
FROM studies
WHERE overall_status = 'COMPLETED'
GROUP BY ___
ORDER BY 시작연도;

**Q4. 두 열로 묶기** — 2024년 이후(`start_date >= '2024-01-01'`)에 시작한 시험을 **연구 유형 · 진행 상태별로** 세세요. `GROUP BY` 뒤에 묶는 열을 적습니다.

In [ ]:
%%sql
-- [26번] Q4
SELECT study_type, overall_status, COUNT(*) AS 건수
FROM studies
WHERE start_date >= '2024-01-01'
GROUP BY ___;

**Q5. HAVING** — 중재 연구(`INTERVENTIONAL`) 중에서 시험이 **2,000건 이상**인 임상 단계만 보세요. `___` 의 조건은 WHERE일까요, HAVING일까요?

In [ ]:
%%sql
-- [27번] Q5
SELECT phase, COUNT(*) AS 건수
FROM studies
WHERE study_type = 'INTERVENTIONAL'
GROUP BY phase
___
ORDER BY 건수 DESC;

**Q6. JOIN** — 일본(`Japan`) 기관에서 진행된 **시험 수**를 세세요. 두 표를 잇는 조건(ON)과 세는 방법을 채웁니다.

In [ ]:
%%sql
-- [28번] Q6
SELECT COUNT(___) AS 일본시험수
FROM studies
JOIN facilities
  ON ___
WHERE country = 'Japan';

**Q7. JOIN 뒤에 GROUP BY** — 3상(`PHASE3`) 시험을 **질환별로** 세어, 많은 순으로 위에서 5줄만 보세요.

In [ ]:
%%sql
-- [29번] Q7
SELECT mesh_term, COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE phase = 'PHASE3'
GROUP BY ___
ORDER BY 시험수 DESC
LIMIT 5;

**Q8. LEFT JOIN** — 질환 정보가 없는 시험도 빠뜨리지 말고 **전체 시험 수**를 세세요. `___` 에 JOIN의 종류를 적습니다.

In [ ]:
%%sql
-- [30번] Q8
SELECT COUNT(DISTINCT studies.nct_id) AS 시험수
FROM studies
___ JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id;

> 그냥 JOIN이면 31,634건 — 37,509건 중 질환 정보가 없는 5,875건이 빠집니다.

## 5. 정리 — 확인 습관 · 용어 · 요약

### 집계와 JOIN을 쓸 때의 확인 습관 — 이 네 가지만 지켜도 대부분의 오류를 막습니다

- ① **무엇을 세는지 말로 정한다** — “시험 건수”인가 “기관 수”인가 “질환 쌍의 수”인가
- ② **JOIN 전후의 행 수를 비교한다** — `COUNT(*)` 와 `COUNT(DISTINCT 키)` 를 같이 본다
- ③ **분모가 줄지 않았는지 본다** — INNER JOIN 때문에 빠진 행이 있는지 LEFT JOIN으로 확인
- ④ **평균은 중앙값과 함께 본다** — 극단값 하나가 결과를 바꿀 수 있다

### 8주차 용어 정리 — 이번 시간에 새로 나온 말

| 용어 | 예시 | 뜻 (한 줄) |
|---|---|---|
| 집계 함수 | `AVG(enrollment)` | 여러 행을 하나의 값으로 만드는 함수 |
| COUNT(*) | `COUNT(*)` | 모든 행의 수 (NULL 포함) |
| COUNT(DISTINCT 열) | `COUNT(DISTINCT phase)` | 서로 다른 값의 개수 |
| MEDIAN | `MEDIAN(enrollment)` | 중앙값. 가운데 값이라 극단값에 덜 흔들림 |
| GROUP BY | `GROUP BY phase` | 같은 값끼리 묶는 절. 결과 1행 = 그룹 1개 |
| CASE WHEN | `CASE WHEN … THEN … END` | 조건에 따라 다른 값을 내놓기. 엑셀의 IF와 같음 |
| COALESCE | `COALESCE(phase, '미기재')` | 빈 값(NULL)을 정해 둔 값으로 바꾸기 |
| HAVING | `HAVING COUNT(*) >= 10` | 묶은 뒤의 그룹에 조건을 거는 절 |
| JOIN | `JOIN b ON …` | 양쪽에 모두 있는 행만 연결 (= INNER JOIN) |
| LEFT JOIN | `LEFT JOIN f ON …` | 왼쪽 표는 모두 남기고 연결 |
| ON | `ON s.nct_id = c.nct_id` | 두 표를 어떤 열로 맞출지 적는 부분 |
| 별칭 | `studies AS s` | 표나 열에 짧은 이름 붙이기 (AS) |

**별칭** — `studies AS s` 처럼 쓰면 `studies.nct_id` 대신 `s.nct_id` 로 짧게 씁니다.

In [ ]:
%%sql
-- [31번] 표 별칭 studies AS s · facilities AS f → 34,810
SELECT COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN facilities AS f
  ON s.nct_id = f.nct_id;

## 오늘 배운 것

- GROUP BY 결과의 한 행이 무엇인지 말로 확인한다
- WHERE는 묶기 전, HAVING은 묶은 뒤 조건이다
- 처리는 FROM → WHERE → GROUP BY → HAVING → SELECT 순서다
- CASE WHEN으로 구간을 나누고, COALESCE로 빈 값에 이름을 붙인다
- JOIN 뒤에는 행 수가 변한다. COUNT(DISTINCT 키)로 확인한다

다음 시간: 9강. 의료데이터 SQL 실습 — 분석 질문을 SQL로 바꾸기, 과제 1 제출

In [ ]:
진행확인()   # [32번]